# 🔍 Deblur/Unblur Image - Training on Google Colab

## 1️⃣ Mount Google Drive
Việc Mount Drive là BẮT BUỘC để lưu trữ model checkpoint. Colab sẽ tự xóa dữ liệu khi bạn tắt tab.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2️⃣ Tải Mã Nguồn Từ GitHub
Chúng ta sẽ kéo (clone) code trực tiếp từ kho GitHub của bạn. Bất cứ khi nào bạn sửa code trên máy tính và push lên Github, bạn chỉ cần chạy lại cell này để lấy code mới nhất.

In [ ]:
import os

# Luôn quay về thư mục gốc của Colab trước khi xử lý
%cd /content

# Xóa thư mục cũ (nếu bạn chạy lại cell này để cập nhật code)
!rm -rf ThiGiacMayTinh

# Clone code từ GitHub của bạn
!git clone https://github.com/hnihTyoB/ThiGiacMayTinh.git

# Di chuyển vào thư mục dự án
%cd ThiGiacMayTinh

# QUAN TRỌNG: Gỡ bỏ bản basicsr lỗi thời nếu có trên Colab
!pip uninstall -y basicsr

# Cài đặt các thư viện cần thiết
!pip install -q lmdb pyyaml tb-nightly tqdm gdown

print('\n[✓] Dự án đã sẵn sàng!')

### 3️⃣ Face
Nếu bạn train Face, hãy upload ảnh gốc (sharp) lên thư mục Drive của bạn, sau đó copy vào môi trường Colab và chạy script tạo blur.

In [ ]:
# 1. Tạo thư mục đích
!mkdir -p datasets/face/train/sharp

# 2. Lấy zip từ Drive (Copy từ Drive đã mount hoặc Tải trực tiếp bằng gdown)
import os
if os.path.exists('/content/drive/MyDrive/celeba_hq_256.zip'):
    print('Detected celeba_hq_256.zip in Google Drive. Copying...')
    !cp /content/drive/MyDrive/celeba_hq_256.zip /content/
else:
    print('celeba_hq_256.zip not found in Google Drive. Downloading directly via gdown...')
    !gdown 1pRr5nMiyRjNwHLqboGXJyzMfQUKthhAz -O /content/celeba_hq_256.zip

# 3. Giải nén vào thư mục sharp
!unzip -q /content/celeba_hq_256.zip -d datasets/face/train/sharp/

# 4. Dùng lệnh 'mv' di chuyển toàn bộ ảnh từ thư mục con ra đúng vị trí
!mv datasets/face/train/sharp/celeba_hq_256/* datasets/face/train/sharp/ 2>/dev/null || true
# 5. Dọn dẹp file zip và thư mục nháp cho nhẹ ổ cứng Colab
!rm -f /content/celeba_hq_256.zip
print('[✓] Đã chuyển toàn bộ ảnh sharp vào đúng thư mục: datasets/face/train/sharp/')

# 6. Chạy lệnh tạo blur:
!python scripts/data_preparation/create_blur_dataset.py --input datasets/face/train/sharp/ --output datasets/face/train/blur --task face

### ID Card
Chuẩn bị dữ liệu ID Card tương tự như Face.

In [ ]:
# 1. Tạo thư mục đích
!mkdir -p datasets/idcard/train/sharp

# 2. Lấy zip từ Drive (Copy từ Drive đã mount hoặc Tải trực tiếp bằng gdown)
import os
if os.path.exists('/content/drive/MyDrive/idcardv2.zip'):
    print('Detected idcardv2.zip in Google Drive. Copying...')
    !cp /content/drive/MyDrive/idcardv2.zip /content/
else:
    print('idcardv2.zip not found in Google Drive. Downloading directly via gdown...')
    !gdown 1qwXyltXDjVRBxNWURr4Oj903y7yRjffm -O /content/idcardv2.zip

# 3. Giải nén vào thư mục sharp
!unzip -q /content/idcardv2.zip -d datasets/idcard/train/sharp/

# Nếu giải nén xong nó nằm trong thư mục con train_frames, hãy di chuyển nó ra:
!mv datasets/idcard/train/sharp/train_frames/* datasets/idcard/train/sharp/ 2>/dev/null || true
!rm -rf datasets/idcard/train/sharp/train_frames 2>/dev/null || true

# Dọn dẹp
!rm -f /content/idcardv2.zip
print('[✓] Đã chuyển toàn bộ ảnh sharp vào đúng thư mục: datasets/idcard/train/sharp/')

# 4. Chạy lệnh tạo blur:
!python scripts/data_preparation/create_blur_dataset.py --input datasets/idcard/train/sharp/ --output datasets/idcard/train/blur --task idcard

## 4️⃣ Training
Bỏ comment (`#`) ở lệnh bạn muốn chạy. Mặc định đang train Scene.

In [ ]:
# CÁCH 1: TỰ ĐỘNG LƯU VÀO GOOGLE DRIVE
# !mkdir -p /content/drive/MyDrive/DeblurModels
# !rm -rf /content/ThiGiacMayTinh/experiments
# !ln -s /content/drive/MyDrive/DeblurModels /content/ThiGiacMayTinh/experiments

# CÁCH 2: TẢI TRỰC TIẾP TỪ GOOGLE DRIVE
# import os
# !rm -rf /content/ThiGiacMayTinh/experiments
# !gdown 11Ldg72RnldpAmuM8iLgEvMFuEpu-XYhj -O /content/DeblurModels.zip
# !unzip -q /content/DeblurModels.zip -d /content/ThiGiacMayTinh/
# if os.path.exists('/content/ThiGiacMayTinh/DeblurModels'):
#     !mv /content/ThiGiacMayTinh/DeblurModels /content/ThiGiacMayTinh/experiments
# else:
#     !mkdir -p /content/ThiGiacMayTinh/experiments
#     !mv /content/ThiGiacMayTinh/DeblurFace_SwinIR_V2 /content/ThiGiacMayTinh/experiments/ 2>/dev/null || true
#     !mv /content/ThiGiacMayTinh/DeblurIDCard_SwinIR_V2 /content/ThiGiacMayTinh/experiments/ 2>/dev/null || true
# !rm -f /content/DeblurModels.zip

# TRAIN FACE DEBLUR
# !python basicsr/train.py -opt options/train/train_deblur_face.yml

# TRAIN ID CARD DEBLUR
# !python basicsr/train.py -opt options/train/train_deblur_idcard.yml

## 5️⃣ Test & Inference (Tùy chọn)
Kiểm tra thử độ hiệu quả trực tiếp trên Colab.

### 📥 Tải Dữ Liệu Test (ID Card & Face)
Chạy cell này để tải dữ liệu kiểm thử trực tiếp từ Google Drive, giải nén và lưu vào các thư mục `test_images/idcard` và `test_images/face`.

In [ ]:
# 1. Tạo các thư mục đích
import os
os.makedirs('test_images/idcard', exist_ok=True)
os.makedirs('test_images/face', exist_ok=True)

# 2. Tải file zip idcard
if not os.path.exists('/content/idcard_test.zip'):
    print('Downloading idcard test zip...')
    !gdown 1Wq028tgNrhKgj8Jg-kE1RpKCcfuRBL4O -O /content/idcard_test.zip

# 3. Tải file zip face
if not os.path.exists('/content/face_test.zip'):
    print('Downloading face test zip...')
    !gdown 1i_7nUwFLXx19hBqgb69tj6AN6wTJt_bS -O /content/face_test.zip

# 4. Giải nén idcard
print('Extracting idcard test images...')
!unzip -q -o /content/idcard_test.zip -d test_images/idcard/
for sub in os.listdir('test_images/idcard'):
    sub_path = os.path.join('test_images/idcard', sub)
    if os.path.isdir(sub_path):
        for f in os.listdir(sub_path):
            os.rename(os.path.join(sub_path, f), os.path.join('test_images/idcard', f))
        os.rmdir(sub_path)

# 5. Giải nén face
print('Extracting face test images...')
!unzip -q -o /content/face_test.zip -d test_images/face/
for sub in os.listdir('test_images/face'):
    sub_path = os.path.join('test_images/face', sub)
    if os.path.isdir(sub_path):
        for f in os.listdir(sub_path):
            os.rename(os.path.join(sub_path, f), os.path.join('test_images/face', f))
        os.rmdir(sub_path)

# 6. Dọn dẹp
!rm -f /content/idcard_test.zip /content/face_test.zip
print('[✓] Đã tải và giải nén thành công vào test_images/idcard và test_images/face!')

In [ ]:
import os
os.makedirs('test_images', exist_ok=True)
os.makedirs('results', exist_ok=True)

TASK = 'idcard'
MODEL_PATH = 'experiments/DeblurIDCard_SwinIR_V2/models/net_g_latest.pth'

# Upload 1-2 ảnh mờ vào thư mục test_images/ bên thanh bên trái trước khi chạy lệnh này
if os.path.exists(MODEL_PATH):
    !python inference/inference_deblur.py \
        --input test_images/{TASK} \
        --output results/{TASK} \
        --model_path {MODEL_PATH} \
        --task {TASK}
else:
    print(f'⚠️ Chưa có model tại: {MODEL_PATH}. Hãy train trước!')